# WiFi CSI HAR — training on Colab

Heavy training runs here; the laptop has no CUDA GPU. Everything else
(collection, QA, dataset assembly, the realtime demo) stays local.

**Before running:** set the runtime to a GPU —
*Runtime -> Change runtime type -> Hardware accelerator: GPU*.

The repo is private and the dataset is ~200 MB, so both arrive via Google
Drive. Nothing here needs a GitHub token or any other credential.


## 1. Mount Drive

Put the project on Drive once, so the layout is:

```
MyDrive/wifi-csi-har/          <- the repo (git clone or a copy)
MyDrive/wifi-csi-har-data/     <- assembled .npz datasets
```

Datasets are git-ignored, so keep them OUT of the repo folder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Install the package

Editable install so edits synced from the laptop take effect on the next
run without reinstalling.


In [ ]:
REPO = '/content/drive/MyDrive/wifi-csi-har'
DATA_DIR = '/content/drive/MyDrive/wifi-csi-har-data'

!test -d {REPO} || echo 'REPO not found - check the path above'
!pip install -q -e {REPO}[ml]


## 3. Confirm the GPU is actually being used

`resolve_device` picks CUDA > MPS > CPU. If this prints `cpu` the runtime
has no GPU attached and training will be as slow as the laptop — fix the
runtime type before going further.


In [ ]:
import torch
from csihar.train import resolve_device

print('torch      :', torch.__version__)
print('cuda avail :', torch.cuda.is_available())
print('device     :', resolve_device('auto'))
if torch.cuda.is_available():
    print('gpu        :', torch.cuda.get_device_name(0))


## 4. Load the dataset

Assemble locally (`csihar.dataset.assemble_dataset` -> `save_dataset`) and
upload the `.npz`. Assembly is CPU-bound preprocessing and gains nothing
from a GPU, so there is no reason to do it here.


In [ ]:
from pathlib import Path
from csihar.dataset import load_dataset, LABEL_NAMES
import numpy as np

DATASET = Path(DATA_DIR) / 'pilot_v0.npz'
ds = load_dataset(DATASET)
print('X', ds.X.shape, ds.X.dtype)
print('subjects   :', sorted(set(ds.subjects.tolist())))
print('sessions   :', len(set(ds.sessions.tolist())))
for i in np.unique(ds.y):
    print(f'  {LABEL_NAMES[i]:<12} {int((ds.y == i).sum())} windows')


## 5. Train

`device='auto'` finds the GPU. The checkpoint is always written back to
CPU tensors so it loads on the laptop for the realtime demo.

**On splits:** `random` leaks — overlapping windows land on both sides.
Report it only alongside `cross-session` and `cross-subject`. Those need
activities recorded in >= 2 sessions and >= 2 subjects respectively; with
fewer, the split is degenerate and its accuracy is meaningless rather than
bad (`dataset.split_coverage_note` will say so).


In [ ]:
from csihar.train import TrainConfig, train_model

cfg = TrainConfig(
    model='cnn',           # or 'cnn_lstm'
    split='random',        # 'cross-session' | 'cross-subject'
    epochs=60,
    batch_size=32,
    lr=1e-3,
    seed=0,
    device='auto',
    results_csv=f'{DATA_DIR}/results/dl.csv',
    checkpoints_dir=f'{DATA_DIR}/checkpoints',
    figures_dir=f'{DATA_DIR}/figures',
)
report, checkpoint = train_model(ds, cfg)
print(report)
print('checkpoint:', checkpoint)


## 6. All three splits, both models

The defensible table for the report. Degenerate splits are skipped with a
note rather than silently logged as a real number.


In [ ]:
import warnings
from csihar.dataset import split_coverage_note

rows = []
for model in ('cnn', 'cnn_lstm'):
    for split in ('random', 'cross-session', 'cross-subject'):
        subjects = sorted(set(ds.subjects.tolist()))
        if split == 'cross-subject' and len(subjects) < 2:
            print(f'skip {model}/{split}: only {len(subjects)} subject(s)')
            continue
        cfg = TrainConfig(
            model=model, split=split, epochs=60, seed=0, device='auto',
            test_subject=subjects[-1] if split == 'cross-subject' else None,
            results_csv=f'{DATA_DIR}/results/dl.csv',
            checkpoints_dir=f'{DATA_DIR}/checkpoints',
            figures_dir=f'{DATA_DIR}/figures',
        )
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            report, _ = train_model(ds, cfg)
        degenerate = any('DEGENERATE' in str(w.message) for w in caught)
        rows.append((model, split, report.accuracy, report.macro_f1,
                     report.falling_recall, degenerate))

print(f"{'model':<10}{'split':<16}{'acc':>8}{'macroF1':>9}{'fall_rec':>10}  note")
for m, s, a, f1, fr, deg in rows:
    fr = '-' if fr is None else f'{fr:.3f}'
    print(f'{m:<10}{s:<16}{a:8.3f}{f1:9.3f}{fr:>10}  '
          f"{'DEGENERATE - not a result' if deg else ''}")


## 7. Bring the results home

`results/dl.csv` and `checkpoints/` are already on Drive. Sync them back to
the repo so the results CSV stays committed and the checkpoint can drive
the local realtime demo:

```bash
cp ~/Drive/wifi-csi-har-data/results/dl.csv experiments/results/dl.csv
cp ~/Drive/wifi-csi-har-data/checkpoints/*.pt experiments/checkpoints/
```

Then locally:

```bash
.venv/bin/python -m csihar.dashboard --checkpoint experiments/checkpoints/cnn_random_0.pt --replay datasets/raw/<session> --speed 1
```
